In [1]:
%load_ext autoreload
%autoreload 2
cmd = 'S'
import sys
parent_dir = "C:/Users/yuhang.hou/projects/pipeline/data_pipeline/bo"
import json
import pandas as pd
import os
import pickle
sys.path.append(parent_dir)
from backtester import *
with open('last_trading_day.json', 'r') as f:
    last_trading_days = json.load(f)
    ltds = {k:pd.to_datetime(v) for k,v in last_trading_days.items() if 'M' not in k}
with open('business_days.json', 'r') as f:
    business_days = json.load(f)
    business_days = [pd.to_datetime(i) for i in business_days]
    business_days = list(set(business_days))
    business_days = sorted(business_days)


def process_results(results):
    res =[]
    for date, data in results.items():
        temp= { }
        temp['date'] = date
        temp['level'] = data['level']
        temp['tc'] = data['tc']
        res.append(temp)
    res = pd.DataFrame(res)
    res = res.set_index('date')
    res['tc_cumsum'] = res['tc'].cumsum()
    return res

def load_future_data(data_path = './data/SM',values = 'close'):
    files = [f for f in os.listdir(data_path) if f.endswith('.csv')]
    dfs = (pd.read_csv(os.path.join(data_path, file)).assign(
        contract=file.replace('.csv', ''),
        date=lambda df: pd.to_datetime(df['date'])
    ) for file in files)

    big_df = pd.concat(dfs, ignore_index=True)
    pivot_df = big_df.pivot(index='date', columns='contract', values= values)

    return pivot_df

contract_df = load_future_data(f'C:/Users/yuhang.hou/projects/pipeline/data/{cmd}',['close'])
contract_df = contract_df.ffill()
# contract_df.head()

contract_df.dropna(how='all', inplace=True)
start_date = '2013-01-09'
end_date = '2025-09-20'

FileNotFoundError: [Errno 2] No such file or directory: 'last_trading_day.json'

In [ ]:
skip_dates = set(
    [
        pd.to_datetime('2001-01-15'),
        pd.to_datetime('2001-02-19'),
        pd.to_datetime('2004-12-31'),
        pd.to_datetime('2005-11-24'),
        pd.to_datetime('2009-11-26'),
        pd.to_datetime('2025-01-01'),
        pd.to_datetime('2025-09-20'),
        pd.to_datetime('2025-09-21'),

    ]
)
data =   vol_series = pd.read_csv('C:/Users/yuhang.hou/projects/pipeline/data/series/BO/BO_0_3.csv')
data['date'] = pd.to_datetime(data['date'])
data = data[data['close'].isna()]
print(len(data))
skip_dates = skip_dates.union(set(data['date']))

business_days = sorted(list(set(business_days)-skip_dates) )


In [ ]:
roll_schedules = [
    'KKNNQUZZZZF*H*',
    # 'KKNNQUZZZZF*H*',
    # 'KNNQUZZZZF*H*K*',
    # 'NNQUZZZZF*H*K*K*',
    # 'NQUZZZZF*H*K*K*N*',
    # 'QUZZZZF*H*K*K*N*N*',
    # 'UZZZZF*H*K*K*N*N*Q*',
    # 'ZZZZF*H*K*K*N*N*Q*U*',
#     'ZZZF*H*K*K*M*M*Q*U*Z*',
#     'ZZZF*H*K*K*M*M*Q*U*Z*Z*',
#     'ZF*H*K*K*M*M*Q*U*Z*Z*Z*',
#     'F*H*K*K*M*M*Q*U*Z*Z*Z*Z*',
]

In [ ]:
final_df = pd.DataFrame()

####  Zscire from mm and cit
'mm_zscore_speed', 'legacy_non_com_zscore_speed',
       'cit_non_com_zscore_speed', 'cit_zscore_speed', 'mm_desea_zscore_speed',
       'mm_net_share_desea_zscore_speed'

In [ ]:
import numpy as np
for i,roll_schedule in enumerate(roll_schedules):
    config = {
        'start_date': start_date,
        'end_date': end_date,
        'roll_start': -10,
        'roll_schedule':roll_schedule,
        'roll_dates': 1,
        'max_position': 1,
        'longshort': 1,
        'roll_style': 'monthly',  
        'cost_type':'percentage',
        'slippage': 0.0005,
        'commission': 0,
        'roll_out' : 1,
        'vol_target': 5000,
        'round': 0,
        'max_daily_volume': 35,
        'lookback':35,
        'signal_col': 'mm_zscore',
        'vol_series':'C:/Users/yuhang.hou/projects/pipeline/data/series/SM/SM_0_3.csv',

    }

    backtest = SignalBacktester(
        data = contract_df,
        config = config,
        trading_days=business_days,
        last_trading_day=ltds,
        signals_delay = 0,
    )
        
    results = backtest.run_backtest()
    res = process_results(results)
    with open(f'strategy/config/s_m_{i}_signal.json', 'w', encoding='utf-8') as f:
        json.dump(config, f)
    with open(f'strategy/backtest/s_m_{i}_signal.pkl', 'wb') as f:
        pickle.dump(results, f)
    # res.plot(figsize=(12,6))
    final_df['mm_zscore'] = res['level']
    # break





In [ ]:
config['signal_col'] = 'mm_zscore_speed'
backtest = SignalBacktester(
        data = contract_df,
        config = config,
        trading_days=business_days,
        last_trading_day=ltds
    )
        
results = backtest.run_backtest()
res = process_results(results)
final_df['mm_zscore_speed'] = res['level']

In [ ]:
config['signal_col'] = 'mm_oi_zscore_speed'
backtest = SignalBacktester(
        data = contract_df,
        config = config,
        trading_days=business_days,
        last_trading_day=ltds
    )
        
results = backtest.run_backtest()
res = process_results(results)
final_df['mm_oi_zscore_speed'] = res['level']

In [ ]:
config['signal_col'] = 'mm_oi_zscore'
backtest = SignalBacktester(
        data = contract_df,
        config = config,
        trading_days=business_days,
        last_trading_day=ltds
    )
        
results = backtest.run_backtest()
res = process_results(results)
final_df['mm_oi_zscore'] = res['level']

In [ ]:
config['signal_col'] = 'legacy_non_com_net_oi_zscore_speed'
backtest = SignalBacktester(
        data = contract_df,
        config = config,
        trading_days=business_days,
        last_trading_day=ltds
    )
        
results = backtest.run_backtest()
res = process_results(results)
final_df['legacy_non_com_net_oi_zscore_speed'] = res['level']
# res.plot(figsize=(12,6))

In [ ]:
config['signal_col'] = 'pmpu_short_share'
backtest = SignalBacktester(
        data = contract_df,
        config = config,
        trading_days=business_days,
        last_trading_day=ltds
    )
        
results = backtest.run_backtest()
res = process_results(results)
final_df['pmpu_short_share'] = res['level']
# res.plot(figsize=(12,6))

In [ ]:
(final_df).plot(figsize=(15,6))
# .loc['2016-01-01':'2018-01-31']

In [ ]:
(final_df['mm_zscore']-final_df['mm_zscore_speed']).plot(figsize=(12,6))

In [ ]:
(final_df[['mm_zscore_speed','mm_zscore']]).plot(figsize=(12,6))

In [ ]:
config['signal_col'] = 'cit_zscore_speed'
backtest = SignalBacktester(
        data = contract_df,
        config = config,
        trading_days=business_days,
        last_trading_day=ltds
    )
        
results = backtest.run_backtest()
res = process_results(results)
final_df['cit_zscore_speed'] = res['level']

# res.plot(figsize=(12,6))

In [ ]:
config['signal_col'] = 'mm_desea_zscore_speed'
backtest = SignalBacktester(
        data = contract_df,
        config = config,
        trading_days=business_days,
        last_trading_day=ltds
    )
        
results = backtest.run_backtest()
res = process_results(results)
final_df['mm_desea_zscore_speed'] = res['level']

In [ ]:
config['signal_col'] = 'mm_net_share_desea_zscore_speed'
backtest = SignalBacktester(
        data = contract_df,
        config = config,
        trading_days=business_days,
        last_trading_day=ltds
    )
        
results = backtest.run_backtest()
res = process_results(results)
final_df['mm_net_share_desea_zscore_speed'] = res['level']

In [ ]:
config['signal_col'] = 'cit_net_oi_zscore_speed'
backtest = SignalBacktester(
        data = contract_df,
        config = config,
        trading_days=business_days,
        last_trading_day=ltds
    )
        
results = backtest.run_backtest()
res = process_results(results)
final_df['cit_net_oi_zscore_speed'] = res['level']

In [ ]:
config['signal_col'] = 'mm_oi_zscore_speed'
backtest = SignalBacktester(
        data = contract_df,
        config = config,
        trading_days=business_days,
        last_trading_day=ltds
    )
        
results = backtest.run_backtest()
res = process_results(results)
final_df['mm_oi_zscore_speed'] = res['level']

In [ ]:
config['signal_col'] = 'legacy_non_com_net_oi_zscore_speed'
backtest = SignalBacktester(
        data = contract_df,
        config = config,
        trading_days=business_days,
        last_trading_day=ltds
    )
        
results = backtest.run_backtest()
res = process_results(results)
final_df['legacy_non_com_net_oi_zscore_speed'] = res['level']

In [ ]:
config['signal_col'] = 'cit_non_com_net_oi_zscore_speed'
backtest = SignalBacktester(
        data = contract_df,
        config = config,
        trading_days=business_days,
        last_trading_day=ltds
    )
        
results = backtest.run_backtest()
res = process_results(results)
final_df['cit_non_com_net_oi_zscore_speed'] = res['level']

In [ ]:
config['signal_col'] = 'legacy_com_zscore_speed'
backtest = SignalBacktester(
        data = contract_df,
        config = config,
        trading_days=business_days,
        last_trading_day=ltds
    )
        
results = backtest.run_backtest()
res = process_results(results)
final_df['legacy_com_zscore_speed'] = res['level']

In [ ]:
config['signal_col'] = 'legacy_non_com_net_oi_zscore'
backtest = SignalBacktester(
        data = contract_df,
        config = config,
        trading_days=business_days,
        last_trading_day=ltds
    )
        
results = backtest.run_backtest()
res = process_results(results)
final_df['legacy_non_com_net_oi_zscore'] = res['level']

In [ ]:
config['signal_col'] = 'mm_oi_zscore'
backtest = SignalBacktester(
        data = contract_df,
        config = config,
        trading_days=business_days,
        last_trading_day=ltds
    )
        
results = backtest.run_backtest()
res = process_results(results)
final_df['mm_oi_zscore'] = res['level']

In [ ]:
final_df.plot(figsize=(12,6))

In [ ]:
(final_df['cit_non_com_zscore_speed']-final_df['legacy_non_com_zscore_speed']).loc['2021-05-02':].plot()#.head(50)

In [ ]:
(final_df['mm_zscore_speed']+0.3*final_df['cit_zscore_speed']).plot(figsize=(12,6))

In [ ]:
(final_df['legacy_non_com_zscore_speed']+0.3*final_df['cit_zscore_speed']).plot(figsize=(12,6))

In [ ]:
final_df.columns

In [ ]:
from utils import *
df = pd.DataFrame()
level_col = []
i=0
df['level_signal'] = load_and_process(f's_m_{i}_signal')['level']

df['level_kf'] = load_and_process(f's_m_{i}_kf')['level']
df['level_sum'] = df['level_kf'] + df['level_signal']
df.plot(figsize=(20,10))

In [ ]:
from utils import *
df = pd.DataFrame()
level_col = []
diff_col = []
for i in range(7):
    df[f'level_{i}'] = load_and_process(f's_m_{i}_kf')['level']/200#/2+load_and_process(f's_m_{i}_kf')['level']/100
    level_col.append(f'level_{i}')
    diff_col.append(f'f{i}-f{0}')
    df[f'f{i}-f{0}'] = df[f'level_{i}'] - df[f'level_{0}']

df[level_col].plot(figsize=(20,10))

In [ ]:
with open(f'strategy/config/c5tc_m_kf.json', 'w', encoding='utf-8') as f:
        json.dump(config, f)

In [ ]:
with open(f'strategy/backtest/c5tc_m_kf.pkl', 'wb') as f:
    pickle.dump(results, f)

In [ ]:

config1 = {
    'start_date': start_date,
    'end_date': end_date,
    'roll_start': -15,
    'roll_schedule': "QMQMQMQUQUQUQZQZQZQH*QH*QH*",
    'roll_dates': 2,
    'max_position': 30,
    'longshort': 1,
    'roll_style': 'quarterly',  
    'cost_type':'percentage',
    'roll_out' : 1,
    'vol_target': 450000,
    'lookback': 5,
    'round':1,
    'vol_series':'C:/Users/yuhang.hou/projects/pipeline/data/series/C5TC/C5TCQ_0_13.csv',
}

backtest = TrendBacktester(
    data = contract_df,
    config = config1,
    trading_days=business_days,
    last_trading_day=ltds
)
    

results1 = backtest.run_backtest()
res1 = process_results(results1)

config2 = {
    'start_date': start_date,
    'end_date': end_date,
    'roll_start': -15,
    'roll_schedule': "QMQMQMQUQUQUQZQZQZQH*QH*QH*",
    'roll_dates': 2,
    'max_position': 30,
    'longshort': 1,
    'roll_style': 'quarterly',  
    'cost_type':'percentage',
    'roll_out' : 1,
    'vol_target': 450000,
    'lookback': 45,
    'round':1,
    'vol_series':'C:/Users/yuhang.hou/projects/pipeline/data/series/C5TC/C5TCQ_0_13.csv',


}

backtest = TrendBacktester(
    data = contract_df,
    config = config2,
    trading_days=business_days,
    last_trading_day=ltds
)
    

results2 = backtest.run_backtest()
res2 = process_results(results2)

(res1+res2).plot()

In [ ]:
with open(f'strategy/config/c5tc_q_kf1.json', 'w', encoding='utf-8') as f:
        json.dump(config1, f)
with open(f'strategy/config/c5tc_q_kf2.json', 'w', encoding='utf-8') as f:
        json.dump(config2, f)
with open(f'strategy/backtest/c5tc_q_kf1.pkl', 'wb') as f:
    pickle.dump(results1, f)
with open(f'strategy/backtest/c5tc_q_kf2.pkl', 'wb') as f:
    pickle.dump(results2, f)

In [ ]:
(res1+res2+res).plot()

In [ ]:
from utils import *
index = 'c5tc_m_kf'
test = load_and_process(index)
test.plot()

In [ ]:
load_and_run_bt(index, 'KF')